# 11 · Embedding 模型：怎么选、何时换

> 重点不是背模型名字，而是建立选择框架：**什么情况下该换 Embedding 模型？**

**本文件覆盖知识点**：BGE / E5 / GTE / Jina / OpenAI / Qwen Embedding / 多语言 / 中文 / Domain-specific Embedding / Embedding 微调（概念）

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 主流模型一张表

| 模型 | 出品 | 特点 | 典型维度 |
|------|------|------|---------|
| **BGE / BGE-M3** | 智源(BAAI) | 中文强、BGE-M3 支持 100+ 语言与稀疏/稠密/多向量 | 1024 |
| **GTE** | 阿里(通义) | 中文&多语均衡；`text-embedding-v3/v4` 即其服务化版本 | 1024(v3) |
| **E5** | Microsoft | 英文/多语通用基准高 | 1024 |
| **Jina Embeddings** | Jina | 8K 长文本嵌入 | 1024 |
| **OpenAI Embeddings** | OpenAI | text-embedding-3 系列 | 3072/1536 |
| **Qwen Embedding** | 阿里百炼 | text-embedding-v1/v2(1536)/v3(1024) | 见上 |

百炼平台上 `text-embedding-v3` 背后即 GTE/千问系服务，中文场景可直接使用。

In [ ]:
# 换模型的“决策清单”：遇到下面任一情况，就该评测/更换 embedding
checklist = [
    '1. 语言不匹配：资料是中文/多语，当前模型偏英文 → 换中文/多语模型',
    '2. 领域专业词差：代码/医疗/法律术语拉不开距离 → 换领域模型或用领域语料微调',
    '3. 相似度分数“压不紧”：相关与无关分数区分度差 → 换更强的模型/带指令前缀的模型',
    '4. 长文本差：超过模型支持长度被截断 → 换长文本嵌入或改切分',
    '5. 检索指标不达标：Recall@K 上不去 → 用离线评测集对比候选模型（第33课）',
]
print('\n'.join(checklist))
print('\n原则：一切以你的评测集上的指标为准，而不是名字或排行榜。')

In [ ]:
# 知识点·真调说明：Embedding 模型选型 —— 给一个“多语言 + 长文 + 内部黑话”场景，让 LLM 现场走一遍判据
# 上面是静态的“换模型信号清单”，这里是把这些判据用在具体场景上、看模型怎么推演。
_llm_live(
    prompt="""我要给企业内部 RAG 选 embedding 模型。场景：资料 60% 中文技术文档、30% 英文产品手册、10% 多语客服邮件，含不少表格和代码片段，部分单篇较长（切分前超过 1 万 token），并有大量内部黑话。请按“语言→领域→文本长度→区分度→离线评测”给我判断步骤，并说明为什么最终要以自己的评测集为准、而不是看排行榜。""",
    system='你是资深 RAG 架构师，回答不超过 8 句、用 ①②③④ 分点，重点给“判据”，不要只罗列模型名。',
    fallback="""① 语言：资料中英多语混杂，优先中文/多语模型（如 GTE、BGE-M3），别选偏英文的。
② 长度与结构：先确认模型支持的目标长度是否覆盖切分后的块，不够就换长文本嵌入或调切分。
③ 区分度：拿“内部黑话”的近义难例做 probe，看相关片段能否压过“貌似相关实则无关”的片段。
④ 评测：在业务语料上跑 Recall@K 对比候选，而不是信排行榜——榜单语料与你领域无关时参考价值有限。""",
    temperature=0.2,
)
print('→ 选型的核心是“判据 + 你自己的数据”，模型名只是入口；下一步就用下面的 5 分钟 probe 真实验证区分度。')

## 2. “判断 embedding 好不好”的关键实验

在少量同类数据上做 A/B：
```text
同一批 问题+相关片段
模型A 能否把“相关片段”排到 Top1？   模型B 呢？
再放几个“貌似相关实则无关”的难例，看谁更分得清。
```

这叫 **probe 小实验**：花 5 分钟就能筛掉明显不合适的模型，再上评测集定论。

In [ ]:
# 知识点·真调说明：Probe 小实验 —— 同一组“问题+真相关+伪装相关”难例，真实对比两个 embedding 模型谁分得清
# 对应讲义 §2：不跑全量评测，先看“真相关片段能否压过伪装相关”这一项，5 分钟完成初筛。
from dotenv import load_dotenv; load_dotenv()
import os
import numpy as np
from dashscope import TextEmbedding

_K = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS = bool(_K) and '你的' not in _K

def _pe(model, texts):
    """真调一次 embedding，返回 L2 归一化后的向量矩阵"""
    r = TextEmbedding.call(model=model, input=texts, api_key=_K)
    if r.status_code != 200:
        raise RuntimeError('Embedding 失败 %s %s' % (r.status_code, r.message))
    a = np.array([e['embedding'] for e in r.output['embeddings']], dtype='float32')
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)

def _cs(a, b):
    return float(a @ b)          # 已归一化，点积即余弦

if _HAS:
    query = '工单超过 SLA 还没处理会怎样'
    related = '服务工单若在约定的 SLA 时限内未解决，会自动升级并计入部门考核'
    hardneg = '本次发版修复了 SLA 时间在界面上显示格式错误的问题（只影响展示层）'
    for m in ('text-embedding-v2', 'text-embedding-v3'):
        try:
            v = _pe(m, [query, related, hardneg])
            s_rel, s_hard = _cs(v[0], v[1]), _cs(v[0], v[2])
            flag = '相关片段排前面 ✅' if s_rel > s_hard else '相关片段没排上 ❌'
            print(f'{m}: 相关={s_rel:.3f}  伪装相关={s_hard:.3f}  区分度={s_rel - s_hard:+.3f}  → {flag}')
        except Exception as e:
            print(f'{m} 调用失败：{e}')
    print('→ 5 分钟 probe 就能看出：在该难例上哪个模型“分得清”。若都压不紧，就该考虑更强模型/微调；初筛通过后再上第 33 课评测集定论。')
else:
    print('未配置 Key，先用常见量级示意（真实运行会输出两个模型在你语料上的实际分数）：')
    print('text-embedding-v2: 相关=0.81  伪装相关=0.78  区分度=+0.03  → 相关片段排前面 ✅（区分度偏紧）')
    print('text-embedding-v3: 相关=0.84  伪装相关=0.71  区分度=+0.13  → 相关片段排前面 ✅（区分度更宽）')
    print('→ probe 的价值：不用跑全量评测，也能快速看出谁把难例分开、谁“压不紧”。')

## 3. 何时该“微调 Embedding”

通用模型在你的**专有名词、缩写、内部黑话**上可能拉不开距离。此时可考虑微调：
- 原理：**对比学习**——把“(问题, 相关文档)”作为正例、(问题, 不相关文档)作为负例训练，让正例向量靠近、负例远离（第 38 课实操思路）；
- 前提：先攒一批标注数据（几百到几千条足够做 LoRA 级微调）；
- 顺序建议：**先换模型/调切分 → 再考虑微调**，微调是锦上添花而非救命稻草。

## 小结

- 模型选择看：语言、领域、长度、区分度、评测指标；
- 用 5 分钟 probe 小实验初筛，用评测集定论；
- 专有领域差再上**对比学习微调**（第 38 课）。